In [15]:
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

In [16]:
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 10
DATA_DIR = './dataset/UTKFace' # Dataset

X = []
y = []

In [17]:
# Class for UTKFace Dataset
class UTKFaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = [f for f in os.listdir(root_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.root_dir, img_name)
        
        # get age from dataset
        try:
            age = float(img_name.split('_')[0])
        except ValueError:
            age = 0.0

        # Load image with PIL (TorchVision prefers PIL Image)
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        # Return as Tensor image and Tensor age
        return image, torch.tensor([age], dtype=torch.float32)

In [18]:
# Transform
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), # Convert to Tensor and Normalize to range 0-1
])

In [19]:
# Create Dataset and DataLoader
dataset = UTKFaceDataset(DATA_DIR, transform=transform)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [20]:
# Build CNN (Convolutional Neural Network) Model
class AgePredictorCNN(nn.Module):
    def __init__(self):
        super(AgePredictorCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 128), # Calculate fully connected layer size based on Conv/Pool layers
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1) # Output 1 is age
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

In [21]:
# Device, Loss Function, Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AgePredictorCNN().to(device)
criterion = nn.MSELoss() # Mean Squared Error (Loss Function)
optimizer = optim.Adam(model.parameters(), lr=0.001) # Optimizer

In [22]:
# Train Model
print(f"Training on {device}...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, ages in train_loader:
        images, ages = images.to(device), ages.to(device)
        
        optimizer.zero_grad() # Zero Gradient
        outputs = model(images)
        loss = criterion(outputs, ages)
        
        loss.backward() # Backpropagation
        optimizer.step() # Update weights
        
        running_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {running_loss/len(train_loader):.4f}")

Training on cpu...
Epoch 1/10, Loss: 330.0439
Epoch 2/10, Loss: 208.8455
Epoch 3/10, Loss: 167.3395
Epoch 4/10, Loss: 147.7178
Epoch 5/10, Loss: 134.6102
Epoch 6/10, Loss: 121.8626
Epoch 7/10, Loss: 113.0981
Epoch 8/10, Loss: 109.7015
Epoch 9/10, Loss: 103.4558
Epoch 10/10, Loss: 97.9550


In [23]:
# Save weight
torch.save(model.state_dict(), 'age_prediction_model.pth')
print("Model saved!")

Model saved!
